# RES-142 — B2 Qwen3 scoring (Colab 2026.04 / G4)

**Prepared only. No runtime was allocated, candidate ZIP uploaded, or reranking run while preparing this notebook.** Keep the authorization fields in the first code cell blank until final source review and a fresh G4 spending authorization.

This notebook accepts only the published candidate ZIP (20 selected queries, 50 frozen RRF candidates each), its bound rights sidecar, and hash-pinned execution helpers. It contains no qrels. Do not add retrieval, embedding, local TEI, fallback model, retries, or CPU fallback.

## Before connecting

Select Colab runtime **2026.04** and one **G4**. Inspect the current G4 CU/hour rate and available balance first. Start an external 25-minute timer at Connect. Enter the freshly approved numeric CU maximum and rate inspection in the first code cell; the notebook's projected CU check is not a platform-enforced cap.

The Drive destination is the existing retrieval run's `b2/` directory. Drive storage authorization is separate. The notebook writes the candidate ZIP and its sidecar before package installation or model downloads; each Drive write is create-only and SHA-256 read-back verified.

In [ ]:
from __future__ import annotations

import hashlib
import json
import time
import traceback
from pathlib import Path

from google.colab import files

RUN_STARTED = time.monotonic()
SESSION_LIMIT_SECONDS = 25 * 60
WORK_LIMIT_SECONDS = 23 * 60

# Keep these gates closed until final source review and a fresh numeric G4 spending authorization.
RUNTIME_VERSION_CONFIRMED = ""
SETUP_AUTHORIZATION = ""
RATE_INSPECTION_AUTHORIZATION = ""
RATE_INSPECTION_GPU = "G4"
RATE_INSPECTION_MAX_CU = 0.0
OBSERVED_CU_PER_HOUR = 0.0
APPROVED_MAX_CU = 0.0
DRIVE_AUTHORIZATION = ""
DRIVE_DESTINATION_CONFIRMATION = ""

EXPECTED_SHA256 = {
    "res142_b2_colab.py": "f9ee4257ade3becf495884b499bf07a8303f416f118708d8ae7ee5bdec9e2ea9",
    "res142_b2_runner.py": "440a34477e6b84f5d1d020e618dc1d9186392763b9df36e639acff4cc3bc5d75",
    "res142_b1_support.py": "d98f7ea12d1df559f11c8be132cc033befb66bebfa13af56e630c870130feedc",
    "b2_execution.py": "2dc3ddd0a5a8aa9044d4eea38106e3920b36f8b741caca00d110ae4383b2701d",
    "res142-colab.lock.txt": "d8a87ad2f1f2a7faccfb96fee9fa859995f61caa7b8a7b845e268888c4c111dc",
    "res142_qwen3_reranker.jinja": "bf7b104801a4e4ad6a147d732e2d5a837a36f708cdbd1b871b864979d3e10acd",
    "res142-b2-candidate-input-v3.zip": "c98d26a7aaf38f35b66203f0d27a73f618800218b1a90c4e322a02ab2c4100ac",
    "rights-confirmation-v3.json": "65980a1e19e6a01444193df55e98fd90a941b4c14b4c83efbd32db3564449d92",
}
REQUIRED_FILES = set(EXPECTED_SHA256)
NORMALIZE_TEXT_FILES = {
    "res142_b2_colab.py", "res142_b2_runner.py", "res142_b1_support.py",
    "b2_execution.py", "res142-colab.lock.txt", "res142_qwen3_reranker.jinja",
}
DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-142"


def require_positive_number(value: object, label: str) -> float:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
        raise PermissionError(f"{label} must be entered as a positive number.")
    return float(value)


try:
    if RUNTIME_VERSION_CONFIRMED != "2026.04":
        raise PermissionError("In Colab, select runtime 2026.04 and confirm it here.")
    if SETUP_AUTHORIZATION != "APPROVE COLAB SETUP":
        raise PermissionError("A fresh APPROVE COLAB SETUP authorization is required.")
    if RATE_INSPECTION_AUTHORIZATION != "APPROVE COLAB RATE INSPECTION":
        raise PermissionError("A fresh G4 rate inspection authorization is required.")
    if RATE_INSPECTION_GPU != "G4":
        raise PermissionError("The rate inspection and scoring target must both be G4.")
    if DRIVE_AUTHORIZATION != "APPROVE COLAB DRIVE STORAGE":
        raise PermissionError("A fresh Google Drive storage authorization is required.")
    if DRIVE_DESTINATION_CONFIRMATION.rstrip("/") != DRIVE_ROOT:
        raise PermissionError(f"Confirm the Drive root exactly as {DRIVE_ROOT!r}.")
    rate_limit = require_positive_number(RATE_INSPECTION_MAX_CU, "Rate-inspection maximum CU")
    observed_rate = require_positive_number(OBSERVED_CU_PER_HOUR, "Observed CU/hour rate")
    approved_max = require_positive_number(APPROVED_MAX_CU, "Approved maximum CU")
    projected_cu = observed_rate * SESSION_LIMIT_SECONDS / 3600
    if projected_cu > approved_max or projected_cu > rate_limit:
        raise PermissionError("The 25-minute CU projection exceeds a fresh numeric approval.")
    print(json.dumps({
        "gpu": "G4",
        "session_limit_seconds": SESSION_LIMIT_SECONDS,
        "work_deadline_seconds": WORK_LIMIT_SECONDS,
        "observed_cu_per_hour": observed_rate,
        "projected_cu": projected_cu,
        "approved_max_cu": approved_max,
        "projection_is_platform_enforced": False,
    }, indent=2))

    uploaded = files.upload()
    if set(uploaded) != REQUIRED_FILES:
        raise ValueError(f"Select exactly the reviewed eight files: {sorted(REQUIRED_FILES)}.")
    input_dir = Path("/content/res142-b2-input")
    if input_dir.exists():
        raise RuntimeError("Refusing to reuse /content/res142-b2-input; start with a fresh runtime.")
    input_dir.mkdir()
    for name, data in uploaded.items():
        if name in NORMALIZE_TEXT_FILES:
            data = data.replace(b"\r\n", b"\n")
        observed = hashlib.sha256(data).hexdigest()
        if observed != EXPECTED_SHA256[name]:
            raise ValueError(f"Uploaded artifact SHA-256 mismatch: {name}")
        path = input_dir / name
        with path.open("xb") as output_file:
            output_file.write(data)

    runner_path = input_dir / "res142_b2_colab.py"
    namespace = {"__name__": "res142_b2_colab_uploaded", "__file__": str(runner_path)}
    source = runner_path.read_text(encoding="utf-8")
    exec(compile(source, str(runner_path), "exec"), namespace)
    result = namespace["main"](
        upload_dir=input_dir,
        approvals={
            "setup": SETUP_AUTHORIZATION,
            "rate_inspection": RATE_INSPECTION_AUTHORIZATION,
            "rate_inspection_gpu": RATE_INSPECTION_GPU,
            "rate_inspection_max_compute_units": rate_limit,
            "gpu": "G4",
            "observed_compute_units_per_hour": observed_rate,
            "approved_max_compute_units": approved_max,
            "drive": DRIVE_AUTHORIZATION,
            "drive_destination_confirmation": DRIVE_DESTINATION_CONFIRMATION,
            "colab_orchestrator_sha256": EXPECTED_SHA256["res142_b2_colab.py"],
        },
        run_started=RUN_STARTED,
        runtime_version_confirmed=RUNTIME_VERSION_CONFIRMED,
    )
    print(json.dumps(result, indent=2, sort_keys=True))
except BaseException as error:
    output_root = Path("/content/res142-b2-preflight")
    output_root.mkdir(parents=True, exist_ok=True)
    already_reported = any(Path("/content/res142-b2/output").glob("failure-*.json"))
    if not already_reported:
        failure_path = output_root / f"failure-{time.time_ns()}.json"
        failure_path.write_text(json.dumps({
            "stage": "notebook_preflight_or_execution",
            "error_type": type(error).__name__,
            "message": str(error)[:2000],
            "traceback": traceback.format_exc()[-12000:],
            "runtime_seconds": time.monotonic() - RUN_STARTED,
        }, indent=2, sort_keys=True), encoding="utf-8")
        try:
            files.download(str(failure_path))
        except Exception:
            pass
    raise

## Frozen scoring profile and stop controls

The source adapter is checked out at exact commit `85024771215947e0315f37a6dd67d1e8d0575eff`; the reviewed B2 adapter helper is overlaid only after its SHA-256 check. The B1-qualified profile pins Qwen/Qwen3-Reranker-0.6B revision `e61197ed45024b0ed8a2d74b80b4d909f1255473`, its matching tokenizer revision, the reviewed Jinja SHA-256, BF16, vLLM 0.31.0/cu129, and batches of 8.

The monotonic work deadline is 23 minutes from the first cell. The last 2 minutes are reserved for process-group termination, Drive flush, and cleanup. The notebook verifies the pinned model files against Hugging Face's immutable LFS SHA-256 or Git blob identity before starting vLLM. It stops on profile, input, rights, GPU-memory, scoring, deadline, or persistence failure; it records the failure and asks the operator to disconnect and delete the runtime manually.

## Output and operator stop

Each completed query writes its 50 reranked rows and a checkpoint with query ID, candidate count, request count, elapsed time, and result SHA-256. The final Drive evidence includes model-file hashes, the exact server argv, package freeze, GPU observations, scores, and cleanup status. The notebook downloads a local evidence archive when Colab permits.

After any outcome, verify the Drive flush status and disconnect/delete the runtime before the 25-minute connection-to-delete limit. This notebook does not authorize itself: the first code cell remains closed until the requested fresh spending authorization is entered.